In [85]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

from tqdm import tqdm

from medmnist import TissueMNIST
from medmnist import INFO

In [86]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"Using: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
else:
    print("CUDA is not available. Using CPU.")

Using: NVIDIA GeForce RTX 4070 Ti SUPER
CUDA: 12.4


In [87]:
info = INFO['tissuemnist']
data_flag = 'tissuemnist'
DataClass = TissueMNIST

task = info['task']  
n_channels = info['n_channels']
n_classes = len(info['label'])

print(f"Number of classes:", n_classes)
print(f"Number of channels: {n_channels}")

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.RandomRotation(degrees=15),  
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

Number of classes: 8
Number of channels: 1


In [88]:
data_train = DataClass(split='train', transform=train_transform, download=True, size=28)
data_test = DataClass(split='test', transform=eval_transform, download=True, size=28)
data_eval = DataClass(split='val', transform=eval_transform, download=True, size=28)

Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz
Using downloaded and verified file: /home/eflammere/.medmnist/tissuemnist.npz


In [89]:
batch_size = 128

dataloader_train = DataLoader(dataset=data_train, batch_size=batch_size, shuffle=True)
dataloader_test = DataLoader(dataset=data_test, batch_size=batch_size, shuffle=False)
dataloader_eval = DataLoader(dataset=data_eval, batch_size=batch_size, shuffle=False)

print(f"\nNumber of images in training dataset: {len(data_train)}")
print(f"Number of images in test dataset: {len(data_test)}")
print(f"Number of images in validation dataset: {len(data_eval)}")



Number of images in training dataset: 165466
Number of images in test dataset: 47280
Number of images in validation dataset: 23640


In [90]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

for param in model.parameters():
    param.requires_grad = True

model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Dropout(0.4),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 8)
)

model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
criterion = nn.CrossEntropyLoss().to(device)
epochs = 1

In [91]:
train_losses, train_accuracies = [], []

for epoch in range(epochs):
    print(f"\nEpoch: {epoch+1}/{epochs}")

    model.train()
    total_loss, total_acc = 0.0, 0.0

    for batch_idx, (inputs, labels) in enumerate(dataloader_train):
        inputs, labels = inputs.to(device), labels.squeeze().to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels.long())
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        batch_acc = (predicted == labels).float().mean().item()
        total_acc += batch_acc

        print(f"Batch {batch_idx+1}/{len(dataloader_train)}\n Loss: {loss.item():.4f} | Acc: {batch_acc:.4f}")

    train_losses.append(total_loss / len(dataloader_train))
    train_accuracies.append(total_acc / len(dataloader_train))

    print(f"\nEpoch {epoch+1} Summary\n Train Loss: {train_losses[-1]:.4f} | Train Acc: {train_accuracies[-1]:.4f}")


    model.eval()
    val_loss, val_acc = 0.0, 0.0
    val_labels, val_predictions, val_probs = [], [], []

    with torch.no_grad():
        for inputs, labels in dataloader_eval:
            print(inputs.shape, labels.shape)
            inputs, labels = inputs.to(device), labels.squeeze().to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels.long())
            val_loss += loss.item()

            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1) 
            
            val_labels.extend(labels.cpu().numpy())
            val_predictions.extend(preds.cpu().numpy())
            val_probs.extend(probs.cpu().numpy()) 

    val_losses.append(val_loss / len(dataloader_eval))
    val_accuracies.append(accuracy_score(val_labels, val_predictions))
    
    val_precision = precision_score(val_labels, val_predictions, average='macro', zero_division=0)
    val_recall = recall_score(val_labels, val_predictions, average='macro', zero_division=0)
    val_f1 = f1_score(val_labels, val_predictions, average='macro', zero_division=0)
    
    try:
        auc_val = roc_auc_score(val_labels, val_probs, multi_class='ovr')
    except ValueError:
        auc_val = 0.0 

    val_precisions.append(val_precision)
    val_recalls.append(val_recall)
    val_f1_scores.append(val_f1)
    val_aucs.append(auc_val)

    print(f"Val Loss: {val_losses[-1]:.4f} | Acc: {val_accuracies[-1]:.4f} | Precision: {val_precision:.4f} | Recall: {val_recall:.4f} | F1: {val_f1:.4f} | AUC: {auc_val:.4f}")

    torch.save(model.state_dict(), "last_model.pth")

print("Saved 'last_model.pth'")

print("Saved 'last_model.pth'")



Epoch: 1/1
Batch 1/1293
 Loss: 2.0281 | Acc: 0.2188
Batch 2/1293
 Loss: 2.0077 | Acc: 0.2109
Batch 3/1293
 Loss: 2.0169 | Acc: 0.2188
Batch 4/1293
 Loss: 2.0030 | Acc: 0.2422
Batch 5/1293
 Loss: 2.0136 | Acc: 0.2500
Batch 6/1293
 Loss: 1.9844 | Acc: 0.2891
Batch 7/1293
 Loss: 1.9827 | Acc: 0.2891
Batch 8/1293
 Loss: 1.9276 | Acc: 0.3828
Batch 9/1293
 Loss: 1.8953 | Acc: 0.3906
Batch 10/1293
 Loss: 1.9058 | Acc: 0.3828
Batch 11/1293
 Loss: 1.8954 | Acc: 0.3438
Batch 12/1293
 Loss: 1.8363 | Acc: 0.4062
Batch 13/1293
 Loss: 1.8280 | Acc: 0.3672
Batch 14/1293
 Loss: 1.8524 | Acc: 0.3516
Batch 15/1293
 Loss: 1.8622 | Acc: 0.4219
Batch 16/1293
 Loss: 1.8520 | Acc: 0.3750
Batch 17/1293
 Loss: 1.8405 | Acc: 0.3906
Batch 18/1293
 Loss: 1.8339 | Acc: 0.3516
Batch 19/1293
 Loss: 1.7778 | Acc: 0.4062
Batch 20/1293
 Loss: 1.7330 | Acc: 0.4375
Batch 21/1293
 Loss: 1.7506 | Acc: 0.3984
Batch 22/1293
 Loss: 1.7506 | Acc: 0.3984
Batch 23/1293
 Loss: 1.6720 | Acc: 0.4844
Batch 24/1293
 Loss: 1.6568 | A

In [92]:
# model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"
# model.load_state_dict(torch.load(model_path, weights_only=True))

test_loss = 0.0
test_labels, test_predictions, test_probs = [], [], []

model.eval()
with torch.no_grad():
    for inputs, labels in dataloader_test:
        inputs = inputs.squeeze(1).to(device)
        labels = labels.squeeze().to(device).long()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1) 
        
        test_labels.extend(labels.cpu().numpy())
        test_predictions.extend(preds.cpu().numpy())
        test_probs.extend(probs.cpu().numpy())  

test_labels = np.array(test_labels)
test_predictions = np.array(test_predictions)
test_probs = np.array(test_probs)

test_accuracy = accuracy_score(test_labels, test_predictions)
test_precision = precision_score(test_labels, test_predictions, average="macro", zero_division=0)
test_recall = recall_score(test_labels, test_predictions, average="macro", zero_division=0)
test_f1 = f1_score(test_labels, test_predictions, average="macro", zero_division=0)

try:
    test_auc = roc_auc_score(test_labels, test_probs, multi_class="ovr")
except ValueError:
    test_auc = 0.0 

print(f"Test Loss: {test_loss / len(dataloader_test):.4f} | Acc: {test_accuracy:.4f} | Precision: {test_precision:.4f} | Recall: {test_recall:.4f} | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")


Test Loss: 1.1287 | Acc: 0.5721 | Precision: 0.5234 | Recall: 0.4760 | F1: 0.4708 | AUC: 0.8880


In [ ]:
# Carregar modelo pré-treinado
model = resnet18(weights=ResNet18_Weights.DEFAULT)

# Carregar pesos treinados
best_model_path = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/experiments/notebooks/classical/best_model.pth"
model.load_state_dict(torch.load(best_model_path, weights_only="True"))

# Remover a camada FC para extração de features
model = nn.Sequential(*list(model.children())[:-1])

# Modo de avaliação
model.to(device)
model.eval()

def extract_features(dataloader, dataset_name, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    features_list, labels_list = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc=f"Extracting Features ({dataset_name})"):
            images = images.to(device)
            features = model(images)

            # Flatten para [batch, 512]
            features = features.view(features.size(0), -1)

            features_list.append(features.cpu().numpy())
            labels_list.append(labels.cpu().numpy())

    features_array = np.concatenate(features_list, axis=0) 
    labels_array = np.concatenate(labels_list, axis=0) 

    np.save(os.path.join(output_dir, f"features_{dataset_name}.npy"), features_array)
    np.save(os.path.join(output_dir, f"labels_{dataset_name}.npy"), labels_array)

    df = pd.DataFrame(features_array)
    df['label'] = labels_array
    df.to_csv(os.path.join(output_dir, f"features_{dataset_name}.csv"), index=False, header=False)


    print(f"{dataset_name} features saved in {output_dir}.")

output_directory = "/home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical"
extract_features(dataloader_train, "MedMNIST", output_directory)


/tmp/ipykernel_603783/2957244615.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path))
Extracting Features (MedMNIST): 100%|

MedMNIST features saved in /home/eflammere/HybridEmbeddingsForBreastCancerDetection/embeddings/classical.
